# Notebook 3.1: Electroosmotic Flow (EOF)

## Objective
Implement electroosmotic flow using the **Helmholtz-Smoluchowski slip boundary condition**:
$$u_{\text{slip}} = \frac{\epsilon \zeta}{\mu} E$$

**Key result to observe:** Flat (plug) velocity profile — completely different from the parabolic Poiseuille profile.

**Problem setup:**
- Same channel geometry as Notebook 2
- Slip BC on walls instead of no-slip
- Electric field drives the flow (no pressure gradient)

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Step 1: Parameters & Mesh

In [ ]:
L   = 1.0   # Channel length
H   = 0.1   # Channel height
mu  = 1.0   # Viscosity

# Electrokinetic parameters
epsilon = 1.0   # Permittivity
zeta    = 0.1   # Zeta potential
E_x     = 1.0   # Applied electric field in x-direction

# Helmholtz-Smoluchowski slip velocity
u_hs = (epsilon * zeta / mu) * E_x
print(f"Helmholtz-Smoluchowski slip velocity: u_HS = {u_hs:.4f}")
print(f"Expected: nearly uniform profile at u_x ≈ {u_hs:.4f}")

mesh = RectangleMesh(Point(0.0, -H/2), Point(L, H/2), 40, 10)
print(f"Mesh: {mesh.num_cells()} cells")

---
## Step 2: Mixed Function Space

In [ ]:
# Taylor-Hood P2-P1 mixed element (same as Notebook 2)
P2 = VectorElement("P", mesh.ufl_cell(), 2)
P1 = FiniteElement("P", mesh.ufl_cell(), 1)
W  = FunctionSpace(mesh, MixedElement([P2, P1]))

print(f"Mixed space DOFs: {W.dim()}")

---
## Step 3: Boundary Conditions

The Helmholtz-Smoluchowski BC replaces no-slip:
$$\mathbf{u}_{\text{wall}} = u_{HS} \, \hat{\mathbf{x}}$$

In the thin double layer limit (typical for microfluidics), the wall velocity is uniform and prescribed.

In [ ]:
tol = 1e-10

def walls(x, on_boundary):
    return on_boundary and (abs(x[1] - H/2) < tol or abs(x[1] + H/2) < tol)

def inlet(x, on_boundary):
    return on_boundary and abs(x[0]) < tol

def outlet(x, on_boundary):
    return on_boundary and abs(x[0] - L) < tol

# Slip BC: tangential (x) velocity = u_HS, normal (y) velocity = 0
u_slip = Constant((u_hs, 0.0))
bc_walls  = DirichletBC(W.sub(0), u_slip,          walls)
bc_inlet  = DirichletBC(W.sub(0), u_slip,          inlet)
bc_outlet = DirichletBC(W.sub(0), u_slip,          outlet)
# Pin pressure at one point to remove nullspace
bc_p      = DirichletBC(W.sub(1), Constant(0.0),   outlet)

bcs = [bc_walls, bc_inlet, bc_outlet, bc_p]
print("Slip BCs applied (Helmholtz-Smoluchowski).")

---
## Step 4: Weak Form & Solve

In [ ]:
(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

# Stokes with no body force (flow driven by wall slip)
a = (mu * inner(grad(u), grad(v)) - p * div(v) + q * div(u)) * dx
L_form = dot(Constant((0.0, 0.0)), v) * dx

w = Function(W)
solve(a == L_form, w, bcs)

u_sol, p_sol = w.split()
print("Solved.")

---
## Step 5: Validate & Compare

EOF gives a **flat (plug) profile** — unlike Poiseuille which is parabolic.

In [ ]:
y_pts  = np.linspace(-H/2 + 1e-6, H/2 - 1e-6, 60)
x_mid  = L / 2.0

u_eof  = np.array([u_sol(x_mid, y)[0] for y in y_pts])
u_poi  = np.array([(1.0/(2*mu)) * (H**2/4 - y**2) for y in y_pts])  # Poiseuille with dp/L=1

# Uniformity: std/mean should be ~0 for plug flow
uniformity = u_eof.std() / u_eof.mean()
print(f"EOF profile: mean={u_eof.mean():.5f}, std={u_eof.std():.2e}")
print(f"Uniformity (std/mean) = {uniformity:.2e}  (→ 0 for perfect plug flow)")
print(f"Slip velocity target:  {u_hs:.5f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(u_eof, y_pts, 'b-',  lw=2.5, label='EOF (plug, FEM)')
ax.plot(u_poi / u_poi.max() * u_hs, y_pts, 'r--', lw=2, label='Poiseuille (normalised)')
ax.axvline(u_hs, color='b', lw=0.8, ls=':', label=f'$u_{{HS}}={u_hs}$')
ax.set_xlabel('$u_x$', fontsize=12)
ax.set_ylabel('$y$',   fontsize=12)
ax.set_title('EOF vs Poiseuille velocity profiles', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/eof_vs_poiseuille.png', dpi=150)
plt.show()

---
## Summary

| Property | Poiseuille | EOF |
|----------|-----------|-----|
| Driver | Pressure gradient | Electric field |
| Wall BC | No-slip ($u=0$) | Slip ($u=u_{HS}$) |
| Profile | Parabolic | Plug (flat) |
| Max velocity | $\Delta p H^2 / 8\mu L$ | $\epsilon\zeta E/\mu$ |

**Exercise:** Vary $\zeta$ and $E$ and confirm $u_{HS}$ scales linearly with both.